# PSF Higher-Order Moments, Modeling Residuals, and Zernike Coefficients — All Bands

For every DP2 visit (all bands: u, g, r, i, z, y) this notebook:

1. Loads from Butler (`refit_psf_star`):
   - **Source moments**: measured from stars — 2nd (Ixx/Ixy/Iyy), 3rd (M30…M03), 4th (M40…M04)
   - **PSF model moments**: what the PSF model predicts at each star — 3rd (`HigherOrderMomentsPSF`) and 4th
2. Computes **PSF modeling residuals** = source − model for all higher-order moments
3. Derives spin-1 (coma), spin-3 (trefoil), and spin-0 (kurtosis ρ₄) from both source moments and residuals
4. Computes **median source moments** and **median residuals per raft** (21 science rafts) per visit
5. Fetches **Zernike coefficients** (Z4–Z11) from ConsDB at all 4 AOS corner detectors
6. Saves one merged Parquet per band to `data/psf_moments_<band>.pq`

**Columns in output DataFrames:**

| Group | Example columns |
|---|---|
| Zernike (median over corners) | `z4`, `z5`, …, `z11` |
| Zernike per corner | `z4_191`, `z7_203`, … |
| Observational conditions | `zenith_distance`, `sky_bg`, `seeing_zenith_500nm` |
| FOV-median source moments | `T`, `e1`, `e2`, `c11`, `c12`, `c31`, `c32`, `rho4` |
| FOV-median residuals | `dT`, `de1`, `de2`, `dc11`, `dc12`, `dc31`, `dc32`, `drho4` |
| Per-raft source moments | `T_R22`, `c11_R01`, … |
| Per-raft residuals | `dT_R22`, `dc11_R01`, … |

In [ ]:
from lsst.daf.butler import Butler
from lsst.obs.lsst import LsstCam
from lsst.summit.utils import ConsDbClient
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os, pickle
from tqdm import tqdm

%matplotlib inline

DATA_DIR = "/sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data"
os.makedirs(DATA_DIR, exist_ok=True)

## Setup: Butler and ConsDB

In [ ]:
repo       = "dp2_prep"
collection = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2"
butler     = Butler(repo, collections=collection)

os.environ["no_proxy"] += ",.consdb"
consdb_url = "http://consdb-pq.consdb:8080/consdb"
cdb_client = ConsDbClient(consdb_url)

# Detector ID → raft name  (e.g. 94 → 'R22')
_camera       = LsstCam.getCamera()
det_to_raft   = {det.getId(): det.getName().split("_")[0] for det in _camera}

SCIENCE_RAFTS = [
    "R01","R02","R03",
    "R10","R11","R12","R13","R14",
    "R20","R21","R22","R23","R24",
    "R30","R31","R32","R33","R34",
    "R41","R42","R43",
]
AOS_DETECTORS  = (191, 195, 199, 203)
ZERNIKE_COLS   = ["z4","z5","z6","z7","z8","z9","z10","z11"]
MOMENT_KEYS    = ["T","e1","e2","c11","c12","c31","c32","rho4","e4_1","e4_2"]
RESID_KEYS     = ["dT","de1","de2","dc11","dc12","dc31","dc32","drho4","de4_1","de4_2"]
ALL_BANDS      = ["u","g","r","i","z","y"]

## Step 1: All-band visit list from Butler

In [ ]:
dsrs = list(butler.registry.queryDatasets("refit_psf_star"))

visit_band_map = {}   # visit_id -> band
for dsr in dsrs:
    visit_band_map[dsr.dataId["visit"]] = dsr.dataId["band"]

# Group by band
band_visits = {b: sorted(v for v, bnd in visit_band_map.items() if bnd == b)
               for b in ALL_BANDS}

print(f"Total visits in DP2: {len(visit_band_map)}")
for b in ALL_BANDS:
    print(f"  {b}-band: {len(band_visits[b])} visits")

## Step 2: Query ConsDB — Zernikes + observational conditions (all visits)

Zernikes are from the 4 AOS corner wavefront sensors (det 191, 195, 199, 203).
One ConsDB query covers all bands; we attach `band` later when merging with PSF data.

In [ ]:
all_visit_ids     = sorted(visit_band_map.keys())
all_visit_ids_str = ", ".join(str(v) for v in all_visit_ids)

zernike_query = f"""
SELECT
    e.exposure_id   AS visit_id,
    e.band,
    ccdvisit1.detector,
    ccdvisit1_quicklook.z4,
    ccdvisit1_quicklook.z5,
    ccdvisit1_quicklook.z6,
    ccdvisit1_quicklook.z7,
    ccdvisit1_quicklook.z8,
    ccdvisit1_quicklook.z9,
    ccdvisit1_quicklook.z10,
    ccdvisit1_quicklook.z11
FROM
    cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
    cdb_lsstcam.ccdvisit1           AS ccdvisit1,
    cdb_lsstcam.exposure            AS e
WHERE
    ccdvisit1.detector IN ({', '.join(str(d) for d in AOS_DETECTORS)})
    AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
    AND ccdvisit1.visit_id    = e.exposure_id
    AND e.exposure_id         IN ({all_visit_ids_str})
"""

raw_z = cdb_client.query(zernike_query).to_pandas()
print(f"ConsDB returned {len(raw_z)} rows, {raw_z['visit_id'].nunique()} unique visits")

In [ ]:
all_visit_ids     = sorted(visit_band_map.keys())
all_visit_ids_str = ", ".join(str(v) for v in all_visit_ids)

property_query = f"""
SELECT
    e.exposure_id AS visit_id,
    e.zenith_distance,
    e.altitude,
    e.azimuth,
    e.dimm_seeing,
    e.humidity,
    e.observation_reason,
    e.wind_dir,
    e.wind_speed
FROM
    cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
    cdb_lsstcam.ccdvisit1 AS ccdvisit1,
    cdb_lsstcam.exposure AS e
WHERE
    ccdvisit1.detector IN ({', '.join(str(d) for d in AOS_DETECTORS)})
    AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
    AND ccdvisit1.visit_id    = e.exposure_id
    AND e.exposure_id         IN ({all_visit_ids_str})
"""

property_df = cdb_client.query(property_query).to_pandas()
print(f"Got {len(property_df)} rows, {property_df['visit_id'].nunique()} unique visits")

In [ ]:
property_df.head(100)

In [ ]:
# Cast Zernike and condition columns to float
obs_cols = []
for col in ZERNIKE_COLS + obs_cols:
    raw_z[col] = pd.to_numeric(raw_z[col], errors="coerce")

# Median Zernikes over the 4 corner detectors per visit
zernike_median = (
    raw_z.groupby("visit_id")[ZERNIKE_COLS].median().reset_index()
)

# Per-corner Zernike pivot:  z4_191, z4_195, ...
zernike_pivot = raw_z.pivot_table(
    index="visit_id", columns="detector", values=ZERNIKE_COLS, aggfunc="first"
)
zernike_pivot.columns = [f"{z}_{det}" for z, det in zernike_pivot.columns]
zernike_pivot = zernike_pivot.reset_index()

# Median observational conditions (any single detector gives the same value per visit)
obs_median = (
    raw_z.groupby("visit_id")[obs_cols].median().reset_index()
)

# Combine into one per-visit Zernike DataFrame
zernike_per_visit = (
    zernike_median
    .merge(zernike_pivot, on="visit_id", how="left")
    .merge(property_df,    on="visit_id", how="left")
)

print(f"Zernike DataFrame: {zernike_per_visit.shape}  visits={zernike_per_visit['visit_id'].nunique()}")
zernike_per_visit.head(3)

## Step 3: Load PSF source moments + PSF model moments from Butler

For each visit we read `refit_psf_star` and extract:

| Quantity | Butler columns | Derived |
|---|---|---|
| Source 2nd moments | `ext_shapeHSM_HsmSourceMoments_xx/xy/yy` | T_src, e1_src, e2_src |
| PSF model 2nd moments | `slot_PSFShape_xx/xy/yy` | T_psf, e1_psf, e2_psf |
| 2nd moment residuals | — | dT = T_src−T_psf, de1, de2 |
| Source 3rd+4th moments | `HigherOrderMomentsSource_*` | c11, c12, c31, c32, rho4 |
| PSF model 3rd+4th moments | `HigherOrderMomentsPSF_*` | c11_psf, c31_psf, rho4_psf |
| Higher-order residuals | — | dc11=c11_src−c11_psf, … |

Per-raft **median** is computed for every quantity.
Results are cached to `data/psf_raw_<band>.pkl` so this cell can be re-run cheaply.

In [ ]:
BUTLER_COLS = [
    "detector",
    # Source 2nd moments (HSM)
    "slot_Shape_xx",
    "slot_Shape_xy",
    "slot_Shape_yy",
    # PSF model 2nd moments
    "slot_PsfShape_xx",
    "slot_PsfShape_xy",
    "slot_PsfShape_yy",
    # Source higher-order moments (3rd order)
    "ext_shapeHSM_HigherOrderMomentsSource_30",
    "ext_shapeHSM_HigherOrderMomentsSource_21",
    "ext_shapeHSM_HigherOrderMomentsSource_12",
    "ext_shapeHSM_HigherOrderMomentsSource_03",
    # Source higher-order moments (4th order)
    "ext_shapeHSM_HigherOrderMomentsSource_40",
    "ext_shapeHSM_HigherOrderMomentsSource_31",
    "ext_shapeHSM_HigherOrderMomentsSource_22",
    "ext_shapeHSM_HigherOrderMomentsSource_13",
    "ext_shapeHSM_HigherOrderMomentsSource_04",
    # PSF model higher-order moments (3rd order)
    "ext_shapeHSM_HigherOrderMomentsPSF_30",
    "ext_shapeHSM_HigherOrderMomentsPSF_21",
    "ext_shapeHSM_HigherOrderMomentsPSF_12",
    "ext_shapeHSM_HigherOrderMomentsPSF_03",
    # PSF model higher-order moments (4th order)
    "ext_shapeHSM_HigherOrderMomentsPSF_40",
    "ext_shapeHSM_HigherOrderMomentsPSF_31",
    "ext_shapeHSM_HigherOrderMomentsPSF_22",
    "ext_shapeHSM_HigherOrderMomentsPSF_13",
    "ext_shapeHSM_HigherOrderMomentsPSF_04",
]


def compute_moments(ixx, ixy, iyy, m30, m21, m12, m03, m40, m31, m22, m13, m04):
    T  = ixx + iyy
    e1 = np.where(T > 0, (ixx - iyy) / T, np.nan)
    e2 = np.where(T > 0, 2.0 * ixy / T,   np.nan)
    return dict(
        T    = T,
        e1   = e1,
        e2   = e2,
        c11  = m30 + m12,
        c12  = m21 + m03,
        c31  = m30 - 3.0 * m12,
        c32  = 3.0 * m21 - m03,
        rho4 = m40 + 2.0 * m22 + m04,
        e4_1 = m40 - m04,
        e4_2 = 2.0 * (m31 + m13),
    )


def raft_medians(arr_dict, mask):
    return {k: float(np.nanmedian(v[mask])) for k, v in arr_dict.items()}


def load_visit(visit, butler_obj=None):
    """Load one visit; butler_obj overrides the global butler."""
    b = butler_obj if butler_obj is not None else butler
    try:
        tbl = b.get("refit_psf_star", visit=visit,
                    parameters={"columns": BUTLER_COLS})
    except Exception as exc:
        return None, str(exc)

    det_ids  = np.array(tbl["detector"])
    raft_ids = np.array([det_to_raft.get(int(d), "unknown") for d in det_ids])

    def col(name):
        return np.array(tbl[name], dtype=float)

    try:
        src = compute_moments(
            col("slot_Shape_xx"),
            col("slot_Shape_xy"),
            col("slot_Shape_yy"),
            col("ext_shapeHSM_HigherOrderMomentsSource_30"),
            col("ext_shapeHSM_HigherOrderMomentsSource_21"),
            col("ext_shapeHSM_HigherOrderMomentsSource_12"),
            col("ext_shapeHSM_HigherOrderMomentsSource_03"),
            col("ext_shapeHSM_HigherOrderMomentsSource_40"),
            col("ext_shapeHSM_HigherOrderMomentsSource_31"),
            col("ext_shapeHSM_HigherOrderMomentsSource_22"),
            col("ext_shapeHSM_HigherOrderMomentsSource_13"),
            col("ext_shapeHSM_HigherOrderMomentsSource_04"),
        )
        psf = compute_moments(
            col("slot_PsfShape_xx"),
            col("slot_PsfShape_xy"),
            col("slot_PsfShape_yy"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_30"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_21"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_12"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_03"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_40"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_31"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_22"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_13"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_04"),
        )
    except Exception as exc:
        return None, f"moment computation failed: {exc}"

    res = {f"d{k}": src[k] - psf[k] for k in MOMENT_KEYS}

    sci_mask = np.isin(raft_ids, SCIENCE_RAFTS)
    record = {}
    record.update(raft_medians(src, sci_mask))
    record.update(raft_medians(res, sci_mask))

    for raft in SCIENCE_RAFTS:
        rmask = (raft_ids == raft)
        if rmask.sum() == 0:
            for k in MOMENT_KEYS: record[f"{k}_{raft}"] = np.nan
            for k in RESID_KEYS:  record[f"{k}_{raft}"] = np.nan
            continue
        for k, v in raft_medians(src, rmask).items(): record[f"{k}_{raft}"] = v
        for k, v in raft_medians(res, rmask).items(): record[f"{k}_{raft}"] = v

    return record, None


In [ ]:
import multiprocessing
import multiprocessing.pool
import gc

NCPU = min(8, multiprocessing.cpu_count())
print(f"Using {NCPU} worker processes  (cpu_count={multiprocessing.cpu_count()})")

# ── Worker: top-level so it is picklable in any multiprocessing mode ─────────
_WORKER_BUTLER       = None
_WORKER_BUTLER_COLS  = None
_WORKER_DET_TO_RAFT  = None
_WORKER_SCI_RAFTS    = None
_WORKER_MOMENT_KEYS  = None
_WORKER_RESID_KEYS   = None

def _worker_init(repo_, collection_, butler_cols_, det_to_raft_, sci_rafts_, moment_keys_, resid_keys_):
    """Runs once per worker process; stores all shared data as globals."""
    global _WORKER_BUTLER, _WORKER_BUTLER_COLS
    global _WORKER_DET_TO_RAFT, _WORKER_SCI_RAFTS, _WORKER_MOMENT_KEYS, _WORKER_RESID_KEYS
    from lsst.daf.butler import Butler as _B
    _WORKER_BUTLER      = _B(repo_, collections=collection_)
    _WORKER_BUTLER_COLS = butler_cols_
    _WORKER_DET_TO_RAFT = det_to_raft_
    _WORKER_SCI_RAFTS   = sci_rafts_
    _WORKER_MOMENT_KEYS = moment_keys_
    _WORKER_RESID_KEYS  = resid_keys_

def _load_visit_worker(visit):
    """Picklable worker: only visit id is sent per task."""
    import numpy as _np

    try:
        tbl = _WORKER_BUTLER.get("refit_psf_star", visit=visit,
                                  parameters={"columns": _WORKER_BUTLER_COLS})
    except Exception as exc:
        return visit, None, f"butler.get failed: {exc}"

    det_ids  = _np.array(tbl["detector"])
    raft_ids = _np.array([_WORKER_DET_TO_RAFT.get(int(d), "unknown") for d in det_ids])

    def col(name):
        return _np.array(tbl[name], dtype=float)

    def _moments(ixx, ixy, iyy, m30, m21, m12, m03, m40, m31, m22, m13, m04):
        T  = ixx + iyy
        e1 = _np.where(T > 0, (ixx - iyy) / T, _np.nan)
        e2 = _np.where(T > 0, 2.0 * ixy / T,   _np.nan)
        return dict(
            T=T, e1=e1, e2=e2,
            c11=m30+m12, c12=m21+m03,
            c31=m30-3*m12, c32=3*m21-m03,
            rho4=m40+2*m22+m04,
            e4_1=m40-m04,
            e4_2=2.0*(m31+m13),
        )

    try:
        src = _moments(
            col("slot_Shape_xx"),
            col("slot_Shape_xy"),
            col("slot_Shape_yy"),
            col("ext_shapeHSM_HigherOrderMomentsSource_30"),
            col("ext_shapeHSM_HigherOrderMomentsSource_21"),
            col("ext_shapeHSM_HigherOrderMomentsSource_12"),
            col("ext_shapeHSM_HigherOrderMomentsSource_03"),
            col("ext_shapeHSM_HigherOrderMomentsSource_40"),
            col("ext_shapeHSM_HigherOrderMomentsSource_31"),
            col("ext_shapeHSM_HigherOrderMomentsSource_22"),
            col("ext_shapeHSM_HigherOrderMomentsSource_13"),
            col("ext_shapeHSM_HigherOrderMomentsSource_04"),
        )
        psf = _moments(
            col("slot_PsfShape_xx"),
            col("slot_PsfShape_xy"),
            col("slot_PsfShape_yy"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_30"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_21"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_12"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_03"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_40"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_31"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_22"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_13"),
            col("ext_shapeHSM_HigherOrderMomentsPSF_04"),
        )
    except Exception as exc:
        del tbl
        return visit, None, f"moment computation failed: {exc}"

    # Free the large Butler table immediately after all columns are extracted
    del tbl

    res = {f"d{k}": src[k] - psf[k] for k in _WORKER_MOMENT_KEYS}

    def _med(ad, mask):
        return {k: float(_np.nanmedian(v[mask])) for k, v in ad.items()}

    sci_mask = _np.isin(raft_ids, _WORKER_SCI_RAFTS)
    record = {}
    record.update(_med(src, sci_mask))
    record.update(_med(res, sci_mask))

    for raft in _WORKER_SCI_RAFTS:
        rmask = (raft_ids == raft)
        if rmask.sum() == 0:
            for k in _WORKER_MOMENT_KEYS: record[f"{k}_{raft}"] = float("nan")
            for k in _WORKER_RESID_KEYS:  record[f"{k}_{raft}"] = float("nan")
            continue
        for k, v in _med(src, rmask).items(): record[f"{k}_{raft}"] = v
        for k, v in _med(res, rmask).items(): record[f"{k}_{raft}"] = v

    # Free intermediate arrays before returning the compact scalar record
    del src, psf, res, det_ids, raft_ids, sci_mask

    return visit, record, None


# ── Main loading loop ─────────────────────────────────────────────────────────
psf_by_band = {}

for band in ALL_BANDS:
    cache_path = os.path.join(DATA_DIR, f"psf_raw_{band}.pkl")

    # if os.path.exists(cache_path):
    #     with open(cache_path, "rb") as fh:
    #         psf_by_band[band] = pickle.load(fh)
    #     print(f"{band}: loaded {len(psf_by_band[band])} visits from cache")
    #     continue

    visits  = band_visits[band]
    records = {}
    errors  = {}

    ctx = multiprocessing.get_context("fork")
    with multiprocessing.pool.Pool(
        processes=NCPU,
        context=ctx,
        initializer=_worker_init,
        initargs=(repo, collection, BUTLER_COLS, det_to_raft,
                  list(SCIENCE_RAFTS), list(MOMENT_KEYS), list(RESID_KEYS)),
        maxtasksperchild=200,   # recycle workers to prevent heap growth
    ) as pool:
        pbar = tqdm(total=len(visits), desc=f"{band}-band")
        for visit_id, rec, err in pool.imap_unordered(_load_visit_worker, visits):
            if rec is None:
                errors[visit_id] = err
                pbar.set_postfix(skipped=len(errors))
            else:
                records[visit_id] = rec
            pbar.update(1)
        pbar.close()

    gc.collect()   # prompt deallocation of worker COW pages after pool closes

    if errors:
        sample = list(errors.items())[:5]
        print(f"\n  {len(errors)}/{len(visits)} visits skipped — first errors:")
        for vid, msg in sample:
            print(f"    visit {vid}: {msg}")

    psf_by_band[band] = records
    with open(cache_path, "wb") as fh:
        pickle.dump(records, fh)
    print(f"{band}: saved {len(records)} visits  ({len(errors)} skipped)  -> {cache_path}")

    del records, errors   # band data now only lives in psf_by_band and on disk
    gc.collect()


## Step 4: Merge PSF moments with Zernikes; save per-band Parquet

In [ ]:
dfs = {}
zernike_visit_set = set(zernike_per_visit["visit_id"].values)

for band in ALL_BANDS:
    records = psf_by_band[band]
    if not records:
        print(f"{band}: no PSF records, skipping")
        continue

    psf_df = pd.DataFrame.from_dict(records, orient="index")
    psf_df.index.name = "visit_id"
    psf_df = psf_df.reset_index()

    # Inner join on Zernike + observational conditions
    merged = zernike_per_visit.merge(psf_df, on="visit_id", how="inner")
    merged.insert(1, "band", band)

    out_path = os.path.join(DATA_DIR, f"psf_moments_{band}.pq")
    merged.to_parquet(out_path, index=False)

    dfs[band] = merged
    print(f"{band}: {len(merged)} visits matched, {merged.shape[1]} columns  → {out_path}")

# Also save one combined Parquet with all bands
# Collapse duplicate visit_ids: last band entry wins (keep='last')
combined = pd.concat(list(dfs.values()), ignore_index=True)
n_before = len(combined)
combined = combined.drop_duplicates(subset=["visit_id"], keep="last").reset_index(drop=True)
n_dupes = n_before - len(combined)
combined.to_parquet(os.path.join(DATA_DIR, "psf_moments_allbands.pq"), index=False)
print(f"\nCombined: {len(combined)} rows ({n_dupes} duplicate visit_ids dropped)  → data/psf_moments_allbands.pq")

In [ ]:
# Load per-band parquets and convert T/dT columns from pixel² → sigma [arcsec]
# sigma = sqrt(T / 2) * pixel_scale;  dsigma = sigma_src - sigma_psf
# Adds new 'sigma' / 'dsigma' (and per-raft 'sigma_R*' / 'dsigma_R*') columns.
# Run this cell instead of the slow Butler loading above.

PIXEL_SCALE = 0.2  # arcsec / pixel (LSSTCam)

def _convert_T_sigma(df):
    df = df.copy()
    t_cols  = [c for c in df.columns if c == 'T'  or c.startswith('T_R')]
    dt_cols = [c for c in df.columns if c == 'dT' or c.startswith('dT_R')]
    # add sigma columns  (sqrt(T / 2) * pixel_scale)
    for col in t_cols:
        sigma_col = 'sigma' + col[1:]   # 'T' -> 'sigma',  'T_R01' -> 'sigma_R01'
        T_pix2 = df[col].values.astype(float)
        df[sigma_col] = np.where(T_pix2 > 0, np.sqrt(T_pix2 / 2.0), np.nan)
    # add dsigma columns  (sigma_src - sigma_psf)
    for dcol in dt_cols:
        paired = dcol[1:]               # 'dT' -> 'T',  'dT_R01' -> 'T_R01'
        if paired not in df.columns:
            continue
        dsigma_col = 'dsigma' + dcol[2:]  # 'dT' -> 'dsigma',  'dT_R01' -> 'dsigma_R01'
        T_src = df[paired].values.astype(float)
        T_psf = T_src - df[dcol].values.astype(float)
        sig_src = np.where(T_src > 0, np.sqrt(T_src / 2.0), np.nan)
        sig_psf = np.where(T_psf > 0, np.sqrt(T_psf / 2.0), np.nan)
        df[dsigma_col] = sig_src - sig_psf
    return df


dfs = {}
for band in ALL_BANDS:
    path = os.path.join(DATA_DIR, f"psf_moments_{band}.pq")
    if not os.path.exists(path):
        print(f"{band}: parquet not found, skipping")
        continue
    df = _convert_T_sigma(pd.read_parquet(path))
    df = df.drop_duplicates(subset=["visit_id"], keep="last").reset_index(drop=True)
    dfs[band] = df
    print(f"{band}: {len(df)} visits  T → sigma/dsigma [arcsec] columns added")

combined = pd.concat(list(dfs.values()), ignore_index=True)
combined = combined.drop_duplicates(subset=["visit_id"], keep="last").reset_index(drop=True)
print(f"\nCombined: {len(combined)} rows  (sigma/dsigma columns available)")


In [ ]:
np.mean(df['sigma_R22'])

In [ ]:
dfs.keys()

In [ ]:
len(dfs['u'])

In [ ]:
len(combined)

In [ ]:
print(np.unique(combined['observation_reason']))

In [ ]:
combined = combined[combined['observation_reason'] == 'field_survey_science']

In [ ]:
print(combined['T_R22'])

## Diagnostic plots

### 5a. Per-band visit counts and moment distributions

In [ ]:
BAND_COLORS = {"u":"#7b2d8b","g":"#1a9641","r":"#d73027","i":"#fc8d59","z":"#4575b4","y":"#8c510a"}

# Source moment FOV distributions per band
plot_moments = ["sigma", "e1", "e2", "c11", "c12", "c31", "c32", "rho4", "e4_1", "e4_2"]
fig, axes = plt.subplots(2, 5, figsize=(22, 7))
fig.suptitle("FOV-median source PSF moments — distribution per band", fontsize=12, fontweight="bold")

MOMENT_LABELS = {"sigma": r"$\sigma$ [arcsec]", "c11": "coma 1", "c12": "coma 2", "c31": "trefoil 1", "c32": "trefoil 2", "rho4": "kurtosis"}
for ax, key in zip(axes.flat, plot_moments):
    for band in ALL_BANDS:
        if band not in dfs: continue
        vals = dfs[band][key].dropna()
        lo, hi = np.nanpercentile(vals, [1, 99])
        ax.hist(vals, bins=60, range=(lo, hi), histtype="step",
                color=BAND_COLORS[band], label=band, density=True)
    ax.set_title(MOMENT_LABELS.get(key, key), fontsize=10)
    ax.set_xlabel("median value")
    ax.set_ylabel("density")
    if key == "sigma":
        ax.legend(fontsize=7, ncol=2)

# Hide unused axes
for ax in axes.flat[len(plot_moments):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("fig/psf_moments_allbands_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig/psf_moments_allbands_dist.png")


In [ ]:
# Modeling residual FOV distributions per band
resid_moments = ["dsigma","de1","de2","dc11","dc12","dc31","dc32","drho4","de4_1","de4_2"]
resid_labels  = [r"$\delta\sigma$ [arcsec]", r"$\delta e_1$", r"$\delta e_2$",
                 r"$\delta c_{11}$", r"$\delta c_{12}$",
                 r"$\delta c_{31}$", r"$\delta c_{32}$", r"$\delta\rho_4$",
                 r"$\delta e_{4,1}$", r"$\delta e_{4,2}$"]

fig, axes = plt.subplots(2, 5, figsize=(22, 7))
fig.suptitle("FOV-median PSF modeling residuals (source − model) — per band", fontsize=12, fontweight="bold")

for ax, key, lbl in zip(axes.flat, resid_moments, resid_labels):
    for band in ALL_BANDS:
        if band not in dfs: continue
        vals = dfs[band][key].dropna()
        lo, hi = np.nanpercentile(vals, [1, 99])
        ax.hist(vals, bins=60, range=(lo, hi), histtype="step",
                color=BAND_COLORS[band], label=band, density=True)
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("residual value")
    ax.set_ylabel("density")
    if key == "dsigma":
        ax.legend(fontsize=7, ncol=2)

# Hide unused axes
for ax in axes.flat[len(resid_moments):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("fig/psf_residuals_allbands_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig/psf_residuals_allbands_dist.png")


### 5a'. Scalar moment magnitudes per band

For each visit we compute the scalar (rotation-invariant) magnitudes:
- **σ** = √(T/2) · pixel_scale [arcsec] (PSF size)
- **|e|** = √(e₁² + e₂²) (ellipticity modulus)
- **|coma|** = √(c₁₁² + c₁₂²) (coma modulus)
- **|trefoil|** = √(c₃₁² + c₃₂²) (trefoil modulus)
- **ρ₄** (fourth-order trace)
- **|e₄|** = √(e₄₁² + e₄₂²) (fourth-moment spin-2 modulus)


In [ ]:
# Scalar (rotation-invariant) moment magnitudes per band
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle(
    "FOV-median scalar PSF moment magnitudes — distribution per band",
    fontsize=12, fontweight="bold"
)

scalar_specs = [
    ("sigma",      r"$\sigma$ [arcsec]",              lambda d: d["sigma"]),
    ("|e|",        r"$|e| = \sqrt{e_1^2+e_2^2}$",    lambda d: np.sqrt(d["e1"]**2 + d["e2"]**2)),
    ("|coma|",     r"$|\mathrm{coma}| = \sqrt{c_{11}^2+c_{12}^2}$",
                                                      lambda d: np.sqrt(d["c11"]**2 + d["c12"]**2)),
    ("|trefoil|",  r"$|\mathrm{trefoil}| = \sqrt{c_{31}^2+c_{32}^2}$",
                                                      lambda d: np.sqrt(d["c31"]**2 + d["c32"]**2)),
    ("rho4",       r"$\rho_4$",                        lambda d: d["rho4"]),
    ("|e4|",       r"$|e_4| = \sqrt{e_{4,1}^2+e_{4,2}^2}$",
                                                      lambda d: np.sqrt(d["e4_1"]**2 + d["e4_2"]**2)),
]

for ax, (key, label, extractor) in zip(axes.flat, scalar_specs):
    for band in ALL_BANDS:
        if band not in dfs:
            continue
        vals = extractor(dfs[band]).dropna()
        if len(vals) < 2:
            continue
        lo, hi = np.nanpercentile(vals, [1, 99])
        ax.hist(vals, bins=60, range=(lo, hi), histtype="step",
                color=BAND_COLORS[band], label=band, density=True)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("FOV-median value")
    ax.set_ylabel("density")
    if key == "sigma":
        ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig("fig/psf_scalar_moments_allbands.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig/psf_scalar_moments_allbands.png")


### 5b. Source moments vs Zernikes (i-band, 2-D histograms)

Same correlation plots as PSF_HM_Zernike.ipynb, now with both source moments and residuals.

In [ ]:
from matplotlib.colors import LogNorm

def prange(col, plo=1, phi=99):
    v = col.dropna()
    return [np.nanpercentile(v, plo), np.nanpercentile(v, phi)]

def hist2d_ax(ax, x, y, xlabel, ylabel, title=""):
    ok = np.isfinite(x) & np.isfinite(y)
    h = ax.hist2d(x[ok], y[ok], bins=80, norm=LogNorm(), cmap="magma",
                  range=[prange(x), prange(y)])
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=9)
    r = np.corrcoef(x[ok], y[ok])[0, 1]
    ax.text(0.05, 0.92, f"r = {r:.3f}", transform=ax.transAxes,
            fontsize=8, color="white", bbox=dict(fc="0.2", alpha=0.6, pad=2))
    return h

# Use i-band for illustration
di = dfs.get("i")
if di is not None:
    pairs = [
        ("z7",  "c11",  "Z7 (coma x)",   "c11 (coma x)"),
        ("z8",  "c12",  "Z8 (coma y)",   "c12 (coma y)"),
        ("z9",  "c31",  "Z9 (trefoil x)","c31 (trefoil x)"),
        ("z10", "c32",  "Z10 (trefoil y)","c32 (trefoil y)"),
        ("z4",  "T",    "Z4 (defocus)",  "T (size)"),
        ("z11", "rho4", "Z11 (spherical)","ρ4 (kurtosis)"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("Source moments vs Zernikes — i-band", fontsize=12, fontweight="bold")
    for ax, (zc, mc, zl, ml) in zip(axes.flat, pairs):
        h = hist2d_ax(ax, di[zc], di[mc], f"{zl} [μm]", ml, f"{zl} vs {ml}")
        plt.colorbar(h[3], ax=ax, label="counts")
    plt.tight_layout()
    plt.savefig("fig/source_vs_zernike_iband.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig/source_vs_zernike_iband.png")

In [ ]:
# Same plots but for residuals vs Zernikes (i-band)
if di is not None:
    resid_pairs = [
        ("z7",  "dc11",  "Z7 (coma x)",    "dc11 residual"),
        ("z8",  "dc12",  "Z8 (coma y)",    "dc12 residual"),
        ("z9",  "dc31",  "Z9 (trefoil x)", "dc31 residual"),
        ("z10", "dc32",  "Z10 (trefoil y)","dc32 residual"),
        ("z4",  "dT",    "Z4 (defocus)",   "dT residual"),
        ("z11", "drho4", "Z11 (spherical)","dρ4 residual"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("PSF modeling residuals vs Zernikes — i-band", fontsize=12, fontweight="bold")
    for ax, (zc, mc, zl, ml) in zip(axes.flat, resid_pairs):
        h = hist2d_ax(ax, di[zc], di[mc], f"{zl} [μm]", ml, f"{zl} vs {ml}")
        plt.colorbar(h[3], ax=ax, label="counts")
    plt.tight_layout()
    plt.savefig("fig/residuals_vs_zernike_iband.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig/residuals_vs_zernike_iband.png")

### 5c. Per-raft focal-plane heatmaps

Median (over all visits) of source moments and residuals for each raft, shown as a 5×5 focal-plane grid.  
Run for all bands.

In [ ]:
RAFT_GRID = [
    [None,  "R01", "R02", "R03", None ],
    ["R10", "R11", "R12", "R13", "R14"],
    ["R20", "R21", "R22", "R23", "R24"],
    ["R30", "R31", "R32", "R33", "R34"],
    [None,  "R41", "R42", "R43", None ],
]

def plot_fp_heatmap(df, moment_key, band, ax, cmap="RdBu_r", symmetric=True):
    """Fill a 5×5 focal-plane heatmap with per-raft medians."""
    grid = np.full((5, 5), np.nan)
    for ri, row in enumerate(RAFT_GRID):
        for ci, raft in enumerate(row):
            if raft is None: continue
            col = f"{moment_key}_{raft}"
            if col in df.columns:
                grid[ri, ci] = df[col].median()

    vmax = np.nanpercentile(np.abs(grid), 98) if symmetric else np.nanpercentile(grid, 98)
    vmin = -vmax if symmetric else np.nanpercentile(grid, 2)

    im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
    for ri, row in enumerate(RAFT_GRID):
        for ci, raft in enumerate(row):
            if raft is None: continue
            v = grid[ri, ci]
            if np.isfinite(v):
                ax.text(ci, ri, f"{v:.3f}", ha="center", va="center",
                        fontsize=6, color="k" if abs(v) < 0.6*vmax else "w")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{moment_key}  ({band})", fontsize=9)
    return im


# Show source moments and residuals including the new e4_1/e4_2 terms
show_keys = ["c11","c12","c31","c32","rho4","e4_1","e4_2",
             "dsigma","dc11","dc31","drho4","de4_1","de4_2"]

for band in ALL_BANDS:
    df_b = dfs.get(band)
    if df_b is None or len(df_b) == 0: continue

    ncols = len(show_keys)
    fig, axes = plt.subplots(1, ncols, figsize=(ncols * 3.0, 3.5))
    fig.suptitle(f"Focal-plane median moments + residuals — {band}-band  ({len(df_b)} visits)",
                 fontsize=11, fontweight="bold")

    for ax, key in zip(axes, show_keys):
        im = plot_fp_heatmap(df_b, key, band, ax, symmetric=False)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    out = f"fig/fp_heatmap_{band}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

### 5c'. Per-raft focal-plane heatmaps — scalar moment magnitudes

Same 5×5 focal-plane grid as above, but now showing the rotation-invariant scalar magnitudes:
T, |e|, |coma|, |trefoil|, ρ₄, |e₄|.  Positive-definite quantities use `viridis`; T uses `plasma`.

In [ ]:
def plot_fp_heatmap_modulus(df, key1, key2, label, band, ax, cmap="viridis"):
    """5x5 heatmap of sqrt(col1^2+col2^2) per raft, median over visits."""
    grid = np.full((5, 5), np.nan)
    for ri, row in enumerate(RAFT_GRID):
        for ci, raft in enumerate(row):
            if raft is None:
                continue
            c1 = f"{key1}_{raft}"
            c2 = f"{key2}_{raft}"
            if c1 in df.columns and c2 in df.columns:
                grid[ri, ci] = np.nanmedian(
                    np.sqrt(df[c1].values**2 + df[c2].values**2)
                )

    finite = grid[np.isfinite(grid)]
    vmin = 0.0
    vmax = np.nanpercentile(finite, 98) if len(finite) else 1.0

    im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
    for ri, row in enumerate(RAFT_GRID):
        for ci, raft in enumerate(row):
            if raft is None:
                continue
            v = grid[ri, ci]
            if np.isfinite(v):
                ax.text(ci, ri, f"{v:.3f}", ha="center", va="center",
                        fontsize=6,
                        color="k" if v < 0.6 * vmax else "w")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{label}  ({band})", fontsize=9)
    return im


scalar_heatmap_specs = [
    (r"$\sigma$ [arcsec]",
     lambda df, band, ax: plot_fp_heatmap(df, "sigma", band, ax,
                                           cmap="plasma", symmetric=False)),
    (r"$|e|$",
     lambda df, band, ax: plot_fp_heatmap_modulus(df, "e1", "e2",
                                                    r"$|e|$", band, ax)),
    (r"$|\mathrm{coma}|$",
     lambda df, band, ax: plot_fp_heatmap_modulus(df, "c11", "c12",
                                                    r"$|\mathrm{coma}|$", band, ax)),
    (r"$|\mathrm{trefoil}|$",
     lambda df, band, ax: plot_fp_heatmap_modulus(df, "c31", "c32",
                                                    r"$|\mathrm{trefoil}|$", band, ax)),
    (r"$\rho_4$",
     lambda df, band, ax: plot_fp_heatmap(df, "rho4", band, ax,
                                           cmap="plasma", symmetric=False)),
    (r"$|e_4|$",
     lambda df, band, ax: plot_fp_heatmap_modulus(df, "e4_1", "e4_2",
                                                    r"$|e_4|$", band, ax)),
]

for band in ALL_BANDS:
    df_b = dfs.get(band)
    if df_b is None or len(df_b) == 0:
        continue

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(
        f"Focal-plane scalar moment magnitudes — {band}-band  ({len(df_b)} visits)",
        fontsize=11, fontweight="bold"
    )

    for ax, (label, plot_fn) in zip(axes.flat, scalar_heatmap_specs):
        im = plot_fn(df_b, band, ax)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    out = f"fig/fp_scalar_heatmap_{band}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

### 5c''. Per-raft moment distributions — all bands

For each scalar moment one 5×5 figure (one panel per raft).  Each panel shows the visit-level distribution for every band as a step histogram.  Corner panels are hidden.  All panels in a given figure share the same x-range (1–99th percentile over all rafts and bands).

In [ ]:
def _safe_modulus(df, k1, k2, raft):
    c1, c2 = f"{k1}_{raft}", f"{k2}_{raft}"
    if c1 not in df.columns or c2 not in df.columns:
        return pd.Series(dtype=float)
    return pd.Series(np.sqrt(df[c1].values**2 + df[c2].values**2))

def _safe_col(df, key, raft):
    col = f"{key}_{raft}"
    return df[col] if col in df.columns else pd.Series(dtype=float)


raft_dist_specs = [
    ("sigma",     r"$\sigma$ [arcsec]",   lambda df, r: _safe_col(df, "sigma", r)),
    ("|e|",       r"$|e|$",                lambda df, r: _safe_modulus(df, "e1",  "e2",  r)),
    ("|coma|",    r"$|coma|$",             lambda df, r: _safe_modulus(df, "c11", "c12", r)),
    ("|trefoil|", r"$|trefoil|$",          lambda df, r: _safe_modulus(df, "c31", "c32", r)),
    ("rho4",      r"$\rho_4$",            lambda df, r: _safe_col(df, "rho4", r)),
    ("|e4|",      r"$|e_4|$",              lambda df, r: _safe_modulus(df, "e4_1", "e4_2", r)),
]

ALL_RAFTS = [r for row in RAFT_GRID for r in row if r is not None]

for key, title, extractor in raft_dist_specs:
    # --- global x-range (1–99th pct across all rafts and bands) ---
    all_vals = np.concatenate([
        extractor(dfs[band], raft).dropna().values
        for band in ALL_BANDS if band in dfs
        for raft in ALL_RAFTS
    ])
    if len(all_vals) == 0:
        print(f"{key}: no data, skipping")
        continue
    xlo, xhi = np.nanpercentile(all_vals, [1, 99])

    fig, axes = plt.subplots(
        5, 5,
        figsize=(13, 13),
        constrained_layout=True,
    )
    fig.suptitle(
        f"Per-raft distribution of {title} — all bands",
        fontsize=13, fontweight="bold"
    )

    legend_ax = None
    for ri, row in enumerate(RAFT_GRID):
        for ci, raft in enumerate(row):
            ax = axes[ri, ci]
            if raft is None:
                ax.set_visible(False)
                continue

            # collect all-band values for this raft (for summary stats)
            raft_all = pd.concat([
                pd.Series(extractor(dfs[band], raft)).dropna()
                for band in ALL_BANDS if band in dfs
            ], ignore_index=True)

            for band in ALL_BANDS:
                if band not in dfs:
                    continue
                vals = pd.Series(extractor(dfs[band], raft)).dropna()
                if len(vals) < 2:
                    continue
                ax.hist(
                    vals, bins=25, range=(xlo, xhi),
                    histtype="step", density=True,
                    color=BAND_COLORS[band], label=band, linewidth=0.9,
                )

            # summary statistics across all bands
            if len(raft_all) >= 2:
                v_median = np.nanmedian(raft_all)
                v_mean   = np.nanmean(raft_all)
                v_p90    = np.nanpercentile(raft_all, 90)
                ymax = ax.get_ylim()[1]
                ax.axvline(v_median, color="k",        lw=1.2, ls="-",  label=f"med {v_median:.3f}")
                ax.axvline(v_mean,   color="dimgray",  lw=1.2, ls="--", label=f"mean {v_mean:.3f}")
                ax.axvline(v_p90,    color="firebrick", lw=1.0, ls=":",  label=f"p90 {v_p90:.3f}")

            ax.set_xlim(xlo, xhi)
            ax.set_title(raft, fontsize=8, pad=2)
            ax.set_yticks([])
            ax.tick_params(axis="x", labelsize=6)
            # force square panel
            ax.set_aspect(1.0 / ax.get_data_ratio(), adjustable="box")
            if legend_ax is None:
                legend_ax = ax

    if legend_ax is not None:
        legend_ax.legend(fontsize=5, ncol=2, loc="upper right")

    safe_key = key.replace("|", "").replace(" ", "_")
    out = f"fig/raft_dist_{safe_key}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")


### 5d. Correlation matrix: source moments + residuals vs Zernikes

Pearson r for each (PSF quantity, Zernike) pair, for each band.

In [ ]:
psf_cols_plot  = ["sigma","e1","e2","c11","c12","c31","c32","rho4","e4_1","e4_2",
                  "dsigma","de1","de2","dc11","dc12","dc31","dc32","drho4","de4_1","de4_2"]
zernike_labels = ["Z4","Z5","Z6","Z7","Z8","Z9","Z10","Z11"]

fig, axes = plt.subplots(2, 3, figsize=(18, 13))
fig.suptitle("Pearson r: PSF moments / residuals vs Zernikes — per band", fontsize=12, fontweight="bold")

bands_with_data = [b for b in ALL_BANDS if b in dfs and len(dfs[b]) > 5]
for ax, band in zip(axes.flat, bands_with_data):
    df_b = dfs[band]
    mat  = np.full((len(psf_cols_plot), 8), np.nan)
    for pi, pc in enumerate(psf_cols_plot):
        for zi, zc in enumerate(ZERNIKE_COLS):
            ok = np.isfinite(df_b[pc]) & np.isfinite(df_b[zc])
            if ok.sum() > 10:
                mat[pi, zi] = np.corrcoef(df_b.loc[ok, pc], df_b.loc[ok, zc])[0, 1]

    im = ax.imshow(mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(8));  ax.set_xticklabels(zernike_labels, fontsize=8)
    ax.set_yticks(range(len(psf_cols_plot))); ax.set_yticklabels(psf_cols_plot, fontsize=8)
    ax.set_title(f"{band}-band  (n={len(df_b)})", fontsize=10)
    plt.colorbar(im, ax=ax, label="Pearson r")

    for pi in range(len(psf_cols_plot)):
        for zi in range(8):
            v = mat[pi, zi]
            if np.isfinite(v):
                ax.text(zi, pi, f"{v:.2f}", ha="center", va="center",
                        fontsize=5.5, color="k" if abs(v) < 0.6 else "w")

for ax in axes.flat[len(bands_with_data):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("fig/corr_matrix_allbands.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig/corr_matrix_allbands.png")

In [ ]:
specific_frame = combined[(combined['visit_id'] > 2025120000355) * (combined['band']=='r')]

In [ ]:
specific_frame

In [ ]:
combined['visit_id']

### 5e. PSF moments vs. azimuth — per band

For each band, scatter + binned-median trend of all 10 source PSF moments
(sigma, e1, e2, coma, trefoil, rho4, e4) as a function of telescope azimuth.

In [ ]:
# PSF moments vs azimuth — one figure per band
plot_moments  = ["sigma", "e1", "e2", "c11", "c12", "c31", "c32", "rho4", "e4_1", "e4_2"]
MOMENT_LABELS = {
    "sigma": r"$\sigma$ [arcsec]", "e1": r"$e_1$", "e2": r"$e_2$",
    "c11":  r"$c_{11}$ (coma)",    "c12": r"$c_{12}$ (coma)",
    "c31":  r"$c_{31}$ (trefoil)", "c32": r"$c_{32}$ (trefoil)",
    "rho4": r"$\rho_4$ (kurtosis)","e4_1": r"$e_{4,1}$", "e4_2": r"$e_{4,2}$",
}
N_BINS = 18  # number of azimuth bins

bands_with_data = [b for b in ALL_BANDS
                   if b in dfs and "azimuth" in dfs[b].columns and len(dfs[b]) > 5]

for band in bands_with_data:
    df_b = dfs[band].dropna(subset=["azimuth"]).copy()
    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    fig.suptitle(f"{band}-band: PSF source moments vs. azimuth  (n={len(df_b)} visits)",
                 fontsize=12, fontweight="bold")

    for ax, key in zip(axes.flat, plot_moments):
        vals = df_b[key]
        az   = df_b["azimuth"]
        ok   = np.isfinite(vals) & np.isfinite(az)
        if ok.sum() < 5:
            ax.set_visible(False)
            continue

        ax.scatter(az[ok], vals[ok], s=12, alpha=0.5,
                   color=BAND_COLORS[band], rasterized=True)

        # binned median trend
        bins    = np.linspace(az[ok].min(), az[ok].max(), N_BINS + 1)
        bin_idx = np.digitize(az[ok], bins) - 1
        bin_meds, bin_errs, bin_cents = [], [], []
        for bi in range(N_BINS):
            mask = (bin_idx == bi)
            n_bi = mask.sum()
            if n_bi >= 3:
                v_bi = vals[ok][mask]
                bin_meds.append(np.nanmedian(v_bi))
                bin_errs.append(np.nanstd(v_bi) / np.sqrt(n_bi))
                bin_cents.append(0.5 * (bins[bi] + bins[bi + 1]))
        if bin_cents:
            ax.errorbar(bin_cents, bin_meds, yerr=bin_errs,
                        color="k", lw=1.8, zorder=5, fmt="-o",
                        markersize=4, capsize=3, label="median ± SE")

        ax.set_title(MOMENT_LABELS.get(key, key), fontsize=10)
        ax.set_xlabel("Azimuth [deg]")
        ax.set_ylabel(key)
        if key == "sigma":
            ax.legend(fontsize=8)

    plt.tight_layout()
    out = f"fig/psf_moments_vs_azimuth_{band}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")


### 5f. PSF moments vs. elevation — per band

For each band, scatter + binned-median trend of all 10 source PSF moments
as a function of telescope elevation (altitude = 90° − zenith distance).

In [ ]:
# PSF moments vs elevation — one figure per band
# ConsDB column is 'altitude' (telescope elevation in degrees)
elev_col = "altitude" if "altitude" in combined.columns else "zenith_distance"

bands_with_data = [b for b in ALL_BANDS
                   if b in dfs and elev_col in dfs[b].columns and len(dfs[b]) > 5]

for band in bands_with_data:
    df_b = dfs[band].dropna(subset=[elev_col]).copy()
    if elev_col == "zenith_distance":
        df_b["_elev"] = 90.0 - df_b["zenith_distance"]
        x_col, x_label = "_elev", "Elevation [deg]"
    else:
        x_col, x_label = elev_col, "Elevation [deg]"

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    fig.suptitle(f"{band}-band: PSF source moments vs. elevation  (n={len(df_b)} visits)",
                 fontsize=12, fontweight="bold")

    for ax, key in zip(axes.flat, plot_moments):
        vals = df_b[key]
        elev = df_b[x_col]
        ok   = np.isfinite(vals) & np.isfinite(elev)
        if ok.sum() < 5:
            ax.set_visible(False)
            continue

        ax.scatter(elev[ok], vals[ok], s=12, alpha=0.5,
                   color=BAND_COLORS[band], rasterized=True)

        # binned median trend
        bins    = np.linspace(elev[ok].min(), elev[ok].max(), N_BINS + 1)
        bin_idx = np.digitize(elev[ok], bins) - 1
        bin_meds, bin_errs, bin_cents = [], [], []
        for bi in range(N_BINS):
            mask = (bin_idx == bi)
            n_bi = mask.sum()
            if n_bi >= 3:
                v_bi = vals[ok][mask]
                bin_meds.append(np.nanmedian(v_bi))
                bin_errs.append(np.nanstd(v_bi) / np.sqrt(n_bi))
                bin_cents.append(0.5 * (bins[bi] + bins[bi + 1]))
        if bin_cents:
            ax.errorbar(bin_cents, bin_meds, yerr=bin_errs,
                        color="k", lw=1.8, zorder=5, fmt="-o",
                        markersize=4, capsize=3, label="median ± SE")

        ax.set_title(MOMENT_LABELS.get(key, key), fontsize=10)
        ax.set_xlabel(x_label)
        ax.set_ylabel(key)
        if key == "sigma":
            ax.legend(fontsize=8)

    plt.tight_layout()
    out = f"fig/psf_moments_vs_elevation_{band}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")


### 5g. Spin-1 (scalar) PSF moment magnitudes vs. azimuth — per band

Same scatter + binned-median plots as 5e, but for the six rotation-invariant
scalars: $\sigma$, $|e|$, $|\mathrm{coma}|$, $|\mathrm{trefoil}|$, $\rho_4$, $|e_4|$.

In [ ]:
# Spin-1 scalar moment magnitudes vs azimuth — one figure per band
scalar_specs = [
    ("sigma",     r"$\sigma$ [arcsec]",
                  lambda d: d["sigma"]),
    ("|e|",        r"$|e| = \sqrt{e_1^2+e_2^2}$",
                  lambda d: np.sqrt(d["e1"]**2 + d["e2"]**2)),
    ("|coma|",     r"$|\mathrm{coma}| = \sqrt{c_{11}^2+c_{12}^2}$",
                  lambda d: np.sqrt(d["c11"]**2 + d["c12"]**2)),
    ("|trefoil|",  r"$|\mathrm{trefoil}| = \sqrt{c_{31}^2+c_{32}^2}$",
                  lambda d: np.sqrt(d["c31"]**2 + d["c32"]**2)),
    ("rho4",       r"$\rho_4$ (kurtosis)",
                  lambda d: d["rho4"]),
    ("|e4|",       r"$|e_4| = \sqrt{e_{4,1}^2+e_{4,2}^2}$",
                  lambda d: np.sqrt(d["e4_1"]**2 + d["e4_2"]**2)),
]

bands_with_data = [b for b in ALL_BANDS
                   if b in dfs and "azimuth" in dfs[b].columns and len(dfs[b]) > 5]

for band in bands_with_data:
    df_b = dfs[band].dropna(subset=["azimuth"]).copy()
    az   = df_b["azimuth"].values

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"{band}-band: scalar PSF moment magnitudes vs. azimuth  (n={len(df_b)} visits)",
                 fontsize=12, fontweight="bold")

    for ax, (key, label, extractor) in zip(axes.flat, scalar_specs):
        vals = extractor(df_b).values
        ok   = np.isfinite(vals) & np.isfinite(az)
        if ok.sum() < 5:
            ax.set_visible(False)
            continue

        ax.scatter(az[ok], vals[ok], s=12, alpha=0.5,
                   color=BAND_COLORS[band], rasterized=True)

        bins    = np.linspace(az[ok].min(), az[ok].max(), N_BINS + 1)
        bin_idx = np.digitize(az[ok], bins) - 1
        bin_meds, bin_errs, bin_cents = [], [], []
        for bi in range(N_BINS):
            mask = (bin_idx == bi)
            n_bi = mask.sum()
            if n_bi >= 3:
                v_bi = vals[ok][mask]
                bin_meds.append(np.nanmedian(v_bi))
                bin_errs.append(np.nanstd(v_bi) / np.sqrt(n_bi))
                bin_cents.append(0.5 * (bins[bi] + bins[bi + 1]))
        if bin_cents:
            ax.errorbar(bin_cents, bin_meds, yerr=bin_errs,
                        color="k", lw=1.8, zorder=5, fmt="-o",
                        markersize=4, capsize=3, label="median ± SE")

        ax.set_title(label, fontsize=10)
        ax.set_xlabel("Azimuth [deg]")
        ax.set_ylabel(key)
        if key == "sigma":
            ax.legend(fontsize=8)

    plt.tight_layout()
    out = f"fig/psf_scalar_moments_vs_azimuth_{band}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")


### 5h. Spin-1 (scalar) PSF moment magnitudes vs. elevation — per band

Same scatter + binned-median plots as 5f, but for the six rotation-invariant scalars.

In [ ]:
# Spin-1 scalar moment magnitudes vs elevation — one figure per band
elev_col = "altitude" if "altitude" in combined.columns else "zenith_distance"

bands_with_data = [b for b in ALL_BANDS
                   if b in dfs and elev_col in dfs[b].columns and len(dfs[b]) > 5]

for band in bands_with_data:
    df_b = dfs[band].dropna(subset=[elev_col]).copy()
    if elev_col == "zenith_distance":
        df_b["_elev"] = 90.0 - df_b["zenith_distance"]
        x_col, x_label = "_elev", "Elevation [deg]"
    else:
        x_col, x_label = elev_col, "Elevation [deg]"
    elev = df_b[x_col].values

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"{band}-band: scalar PSF moment magnitudes vs. elevation  (n={len(df_b)} visits)",
                 fontsize=12, fontweight="bold")

    for ax, (key, label, extractor) in zip(axes.flat, scalar_specs):
        vals = extractor(df_b).values
        ok   = np.isfinite(vals) & np.isfinite(elev)
        if ok.sum() < 5:
            ax.set_visible(False)
            continue

        ax.scatter(elev[ok], vals[ok], s=12, alpha=0.5,
                   color=BAND_COLORS[band], rasterized=True)

        bins    = np.linspace(elev[ok].min(), elev[ok].max(), N_BINS + 1)
        bin_idx = np.digitize(elev[ok], bins) - 1
        bin_meds, bin_errs, bin_cents = [], [], []
        for bi in range(N_BINS):
            mask = (bin_idx == bi)
            n_bi = mask.sum()
            if n_bi >= 3:
                v_bi = vals[ok][mask]
                bin_meds.append(np.nanmedian(v_bi))
                bin_errs.append(np.nanstd(v_bi) / np.sqrt(n_bi))
                bin_cents.append(0.5 * (bins[bi] + bins[bi + 1]))
        if bin_cents:
            ax.errorbar(bin_cents, bin_meds, yerr=bin_errs,
                        color="k", lw=1.8, zorder=5, fmt="-o",
                        markersize=4, capsize=3, label="median ± SE")

        ax.set_title(label, fontsize=10)
        ax.set_xlabel(x_label)
        ax.set_ylabel(key)
        if key == "sigma":
            ax.legend(fontsize=8)

    plt.tight_layout()
    out = f"fig/psf_scalar_moments_vs_elevation_{band}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")
